In [ ]:
%matplotlib inline

In [ ]:
from IPython.display import HTML
import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

# 1D linear shallow water equations in JAX

The shallow water equations are a depth-averaged approximation to the Navier-Stokes equations. The standard 2D formulation makes use of a horizontal velocity $\mathbf{u}:\Omega\times(0,T)\rightarrow\mathbb{R}^2$ and a free surface elevation field $\eta:\Omega\times(0,T)\rightarrow\mathbb{R}$, where we have a spatial domain $\Omega\subset\mathbb{R}^2$ and time period $(0,T)$ for some $T>0$.
The free surface elevation $\eta$ can be interpreted as the perturbation of the water surface from rest.
The water depth at rest is referred to as the *bathymetry*, $b$, and the total water depth is $h=b+\eta$.
The bathymetry is fixed in time, whereas the total water depth is not.

In this mini-project, we apply several simplifications:

1. Consider a 1D spatial domain rather than a 2D spatial domain.
2. Do not consider Coriolis forces due to planetary rotation.
3. Do not consider nonlinear terms.

The non-rotational shallow water equations are given by
$$
\begin{align}
\frac{\partial u}{\partial t}+\nabla\cdot(\nabla\mathbf{u})+g\nabla\eta&=\nabla\cdot(\nu(\nabla\mathbf{u}+(\nabla\mathbf{u})^T)),\\
\frac{\partial\eta}{\partial t}+\nabla\cdot(h\mathbf{u})&=0,
\end{align}
$$
where $\nu$ is a viscosity coefficient, and $g$ is the gravitational acceleration constant.

Restricting to the 1D case and 'linearising', the advection and viscosity terms are dropped and we are left with
$$
\begin{align}
\frac{\partial u}{\partial t}+g\frac{\partial\eta}{\partial x}&=0,\\
\frac{\partial\eta}{\partial t}+b\frac{\partial u}{\partial x}+u\frac{\partial b}{\partial x}&=0,
\end{align}
$$
where now $u:\Omega\times(0,T)\rightarrow\mathbb{R}$.
Note that $\frac{\partial b}{\partial x}$ is constant in time. 

Consider the 1D spatial domain $\Omega=(0,X)$ with $X>0$.

Combining the state as a vector $\mathbf{w}=(u,\eta)$ we have
$$
\frac{\partial\mathbf{w}}{\partial t}
+\begin{bmatrix}0&g\\b&0\end{bmatrix}\frac{\partial\mathbf{w}}{\partial x}
+\begin{bmatrix}0&0\\\frac{\partial b}{\partial x}&0\end{bmatrix}\mathbf{w}
=\boldsymbol0,
$$
which can be shown to be equivalent to a wave equation with wavespeed $\sqrt{gb}$.
We introduce the notation
$$
\underline{\mathbf{A}}=\begin{bmatrix}0&g\\b&0\end{bmatrix},
\quad\underline{\mathbf{B}}=\begin{bmatrix}0&0\\b&0\end{bmatrix}
$$
for conciseness so that
$$
\frac{\partial\mathbf{w}}{\partial t}
+\underline{\mathbf{A}}\frac{\partial\mathbf{w}}{\partial x}
+\frac{\partial\underline{\mathbf{B}}}{\partial x}\mathbf{w}
=\boldsymbol0,
$$

For initial conditions, assume zero velocity $u(x,0)=0,\:\forall x\in(0,X)$ and spatially varying free surface $\eta(x,0)=\eta_0(x)$.

Assume periodic boundary conditions for simplicity: $\mathbf{w}(0,t)=\mathbf{w}(X,t),\:\forall t\in(0,T)$.

Set the spatial and temporal extents.

In [ ]:
X = 400e3
T = 4200

Define the number of points for discretising both space and time.

In [ ]:
nx = 400
nt = 4200

Determine the grid spacing and timestep.

In [ ]:
assert nx > 1
assert nt > 1
x = np.linspace(0, X, nx)
t = np.linspace(0, T, nt)
dx = x[1] - x[0]
dt = t[1] - t[0]
print(f"dx = {dx:.4f}")
print(f"dt = {dt:.4f}")

Define the initial conditions as follows to correspond to the tsunami modelling problem in [1].
We apply a small perturbation to the free surface elevation to see how it propagates across the domain.

In [ ]:
u0 = jnp.zeros_like(x)
eta0 = jnp.maximum(0.4 - ((x - 125e3) / 25e3) ** 2, 0.0)
w0 = jnp.vstack((u0, eta0)).transpose().flatten()

Consider a constant bathymetry (water depth at rest) of $4\,\mathrm{km}$ and the standard gravitational acceleration constant. Note that our implementation supports a spatially varying bathymetry. We also define `g` using JAX syntax so that we can differentiate with respect to it.

In [ ]:
b = jnp.ones(nx) * 4000.0
g = jnp.array([9.81])

In [ ]:
def plot_solution(sol, axes=None, both=False):
    """Plot the solution field.

    :arg sol: the solution array corresponding to both velocity and elevation at a time level
    :kwarg axes: optional matplotlib axes object to plot on
    :kwarg both: logical flag for plotting both solution fields when True or just elevation when False
    """
    if axes is None:
        fig, axes = plt.subplots(figsize=(6, 2))
    if both:
        axes.plot(x / 10e3, sol[0::2], label=r"Velocity, $u$")
    axes.plot(x / 10e3, sol[1::2], label=r"Elevation, $\eta$")
    axes.set_xlabel(r"$x$ [km]")
    if both:
        axes.legend()
        axes.set_ylabel("Magnitude")
    else:
        axes.set_ylabel(r"Elevation, $\eta$ [m]")
        axes.set_ylim([-0.4, 0.4])
    axes.set_xlim([0, 40])
    axes.grid()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 2))
plot_solution(w0, axes=axes[0], both=True)
axes[0].set_title("Initial conditions")
axes[1].set_title("Bathymetry")
axes[1].invert_yaxis()  # Flip the y-axis for the bathymetry plot
axes[1].plot(x / 10e3, b)
axes[1].set_xlabel(r"$x$ [m]")
axes[1].set_ylabel(r"Bathymetry, $b$ [km]")
axes[1].set_xlim([0, 40])
axes[1].grid()

Applying implicit Euler for timestepping gives the approximation
$$
\frac{\mathbf{w}^{k+1}-\mathbf{w}^k}{\Delta t}
+\underline{\mathbf{A}}\frac{\partial\mathbf{w}^{k+1}}{\partial x}
+\frac{\partial\underline{\mathbf{B}}}{\partial x}\mathbf{w}^{k+1}
\approx\boldsymbol0.
$$

Applying a central difference for the spatial derivative gives
$$
\frac{\mathbf{w}^{k+1}_i-\mathbf{w}^k_i}{\Delta t}
+\underline{\mathbf{A}}\frac{\mathbf{w}^{k+1}_{i+1}-\mathbf{w}^{k+1}_{i-1}}{2\Delta x}
+\frac{\underline{\mathbf{B}}_{i+1}-\underline{\mathbf{B}}_{i-1}}{2\Delta x}\mathbf{w}^{k+1}_i
\approx\boldsymbol0.
$$

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Note</b> We choose implicit Euler for the temporal discretisation and central differences for the spatial discretisation because this pairing is known to be stable.

</div>

Setting $c=\frac{\Delta t}{2\Delta x}$ for conciseness and rearranging, we arrive at:
$$
\mathbf{w}^{k+1}_i+c\left(\underline{\mathbf{A}}(\mathbf{w}^{k+1}_{i+1}-\mathbf{w}^{k+1}_{i-1})+(\underline{\mathbf{B}}_{i+1}-\underline{\mathbf{B}}_{i-1})\mathbf{w}^{k+1}_i\right)
\approx\mathbf{w}^k_i
$$

In [ ]:
c = dt / (2 * dx)
print(f"c = {c:.4f}")

This gives rise to the block matrix system
$$
\begin{bmatrix}
\underline{\mathbf{I}}+c(\underline{\mathbf{B}}_2-\underline{\mathbf{B}}_{n_x}) & c\underline{\mathbf{A}} & & & -c\underline{\mathbf{A}}\\
-c\underline{\mathbf{A}} & \underline{\mathbf{I}}+c(\underline{\mathbf{B}}_3-\underline{\mathbf{B}}_1) & c\underline{\mathbf{A}}\\
& -c\underline{\mathbf{A}} & \underline{\mathbf{I}}+c(\underline{\mathbf{B}}_4-\underline{\mathbf{B}}_2) & c\underline{\mathbf{A}}\\
& & \ddots & \ddots & \ddots\\
c\underline{\mathbf{A}} & & & -c\underline{\mathbf{A}} & \underline{\mathbf{I}}+c(\underline{\mathbf{B}}_1-\underline{\mathbf{B}}_{n_x-1})\\
\end{bmatrix}\mathbf{w}^{k+1}
\approx\mathbf{w}^k,
$$
where $\underline{\mathbf{I}}$ is the $2\times2$ identity matrix.
Both block matrices are tridiagonal except that they have additional nonzero entries in the top-right and bottom-left entries due to the periodic boundary conditions.

The matrices are fixed in time so can be precomputed ahead-of-time.
Given the state at timestep $k$, this is a linear system we can solve to approximate the state at timestep $k+1$.

For further convenience, define
$$
\underline{\mathbf{U}}=c\begin{bmatrix}
& \underline{\mathbf{A}} & & &\\
& & \underline{\mathbf{A}}\\
& & & \underline{\mathbf{A}}\\
& & & & \ddots\\
\underline{\mathbf{A}}
\end{bmatrix},
$$
$$
\underline{\mathbf{L}}=-c\begin{bmatrix}
& & & & \underline{\mathbf{A}}\\
\underline{\mathbf{A}}\\
& \underline{\mathbf{A}}\\
& & \ddots\\
& & & \underline{\mathbf{A}}
\end{bmatrix},
$$
and
$$
\underline{\mathbf{D}}=\begin{bmatrix}
\underline{\mathbf{I}}+c(\underline{\mathbf{B}}_2-\underline{\mathbf{B}}_{n_x-1})\\
& \underline{\mathbf{I}}+c(\underline{\mathbf{B}}_3-\underline{\mathbf{B}}_1)\\
& & \ddots\\
& & & \underline{\mathbf{I}}+c(\underline{\mathbf{B}}_1-\underline{\mathbf{B}}_{n_x-1})
\end{bmatrix},
$$
so that we have
$$
(\underline{\mathbf{L}}+\underline{\mathbf{D}}+\underline{\mathbf{U}})\mathbf{w}^{k+1}
\approx\mathbf{w}^k.
$$
We can define these matrices as follows:

In [ ]:
B_diff = jnp.roll(b, -1) - jnp.roll(b, 1)
diagonal = jnp.eye(2 * nx) \
        + c * jnp.diag(jnp.concatenate((B_diff[0:1], jnp.vstack((jnp.zeros(nx-1), B_diff[1:])).transpose().flatten())), k=-1)
upper = c * (jnp.diag(jnp.concatenate((jnp.vstack((jnp.zeros(nx-1), b[:-1])).transpose().flatten(), jnp.zeros(1))), k=1) \
        + jnp.diag(jnp.concatenate((jnp.vstack((g * jnp.ones(nx-2), jnp.zeros(nx-2))).transpose().flatten(), g * jnp.ones(1))), k=3) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), g * jnp.ones(1), jnp.zeros(1))), k=-(2*nx-3)) \
        + jnp.diag(b[-1:], k=-(2*nx-1)))
lower = -c * (jnp.diag(jnp.concatenate((jnp.vstack((b[2:], jnp.zeros(nx-2))).transpose().flatten(), b[1:2])), k=-3) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), jnp.vstack((g * jnp.ones(nx-1), jnp.zeros(nx-1))).transpose().flatten())), k=-1) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), b[0:1], jnp.zeros(1))), k=2*nx-3) \
        + jnp.diag(g * jnp.ones(1), k=2*nx-1))

In [ ]:
lhs_matrix = lower + diagonal + upper

We can plot the sparsity pattern of the matrix as follows:

In [ ]:
plt.spy(lhs_matrix);

Solve the PDE by integrating in time and solving the linear system at each timestep.
We import the `tqdm` utility to display a progress bar.

In [ ]:
from tqdm import tqdm

In [ ]:
def pde_solve(w_init):
    """Solve the PDE.

    :arg w_init: initial condition
    :return: solution trajectory
    """
    # Set initial condition
    w = w_init
    trajectory = [w]
    
    # Do the time integration
    for k, time in enumerate(tqdm(t)):
        w = jnp.linalg.solve(lhs_matrix, w)
        trajectory.append(w)
    return trajectory

In [ ]:
trajectory = pde_solve(w0)

Plot the solution trajectory at a few key times.

In [ ]:
snapshot_times = [525, 1365, 2772, 3255, 4200]
idx = 0
fig, axes = plt.subplots(nrows=5, figsize=(6, 12))
for time, sol in zip(t, trajectory):
    if time >= snapshot_times[idx]:
        plt.text(0.98, 0.95, f"t={time:.0f}", transform=axes[idx].transAxes, fontsize=12, ha="right", va="top")
        plot_solution(sol, axes=axes[idx])
        idx += 1

Produce an animation, too.

In [ ]:
fig, axes = plt.subplots(figsize=(6, 2))
axes.axis([0, 40e3, -0.4, 0.4])
l, = axes.plot([],[])

frame_rate = 25

def animate(i):
    axes.clear()
    plot_solution(trajectory[frame_rate * i], axes=axes)

ani = matplotlib.animation.FuncAnimation(fig, animate, frames=len(trajectory[::frame_rate]))
plt.close()
HTML(ani.to_jshtml())

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
<h2>Mini-project ideas</h2>

1. Use differentiable programming to compute sensitivities of the model with respect to the initial free surface elevation `eta0`. One way to do this is to reformulate the `pde_solve` function so that its argument is `eta0` rather than `w0`. Consider also sensitivities of the model with respect to the gravitational constant `g` and with respect to the bathymetry field `b`. Try out spatially varying bathymetry fields.
2. Conduct a source inversion experiment where you seek to recover the initial free surface elevation `eta0` by optimising the fit of the free surface elevation solution at a set of points against timeseries. Run the model for a given initial condition and extract timeseries data at a set of points. Use these data to define an objective function. Then change the initial condition and see if you can recover your initial choice of `eta0`.

</details>
</div>

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Note on idea 1:</b> The numerical scheme is unlikely to be able to cope with shocks due to sharp gradients in bathymetry so you should avoid 'shelf break' step functions. You should also ensure the bathymetry field is periodic.

<b>Note on idea 2:</b> This is an ill-posed problem! You will likely need to regularise the problem by restricting `eta0` to only be non-zero in a specific region and/or include a regularisation term in your objective function.

</div>

## References

[1] Davis, B. N., & LeVeque, R. J. (2016). Adjoint methods for guiding adaptive mesh refinement in tsunami modeling. In Global tsunami science: Past and future, volume I (pp. 4055-4074). Birkhäuser, Cham.